# P1 폐렴 3분류 — v7 단일 실험 (모델/해상도 바꿔가며)

**사용법:** CELL 2의 `MODEL`, `IMG_SIZE` **두 줄만 바꾸고** Run All.
한 번에 하나 돌리고, 끝나면 그 두 줄만 바꿔 다시 Run.

**돌려볼 조합 (batch16+lr1e-4 고정, 현재 최고 r18_224=0.866 기준):**
- `MODEL='resnet18', IMG_SIZE=288` → 해상도만 ↑
- `MODEL='resnet34', IMG_SIZE=224` → 모델만 ↑
- `MODEL='resnet34', IMG_SIZE=288` → 둘 다 ↑

각 실험은 S3에 `v7/run_{시각}_{MODEL}_{IMG_SIZE}/` 로 자동 저장되고, 마지막에 로컬 다운로드 명령 출력.

### ⚠️ GPU 전환 + download_data 선행 + pneumonia_labels.csv 업로드. 끝나면 Stop.
### (resnet34·288은 r18·224보다 느림: 하나당 15~30분)

## CELL 0 — 디바이스 확인

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device, '| CUDA:', torch.cuda.is_available())
if device.type != 'cuda':
    print('⚠️ GPU 아님 → ml.g4dn.xlarge 로 바꾸세요 (resnet34/288은 CPU면 매우 느림)')

## CELL 1 — 공통 설정 + 데이터 분할 (환자단위 val)

In [ ]:
import io, json, time, os, numpy as np, pandas as pd, boto3
from PIL import Image

VERSION = 'v7'
DATA_BUCKET = 'aws-say3-dataset1'
OUT_BUCKET  = 'aws-say3-student-dataset'
S3_DEV_ROOT = 'students/aws-say3-11/P1/model_dev'
MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]
NUM_CLASSES = 3
CLASS_NAMES = ['NORMAL', 'BACTERIAL_PNEUMONIA', 'VIRAL_PNEUMONIA']
SEED = 42
VAL_RATIO = 0.15
LOCAL_DATA = os.path.abspath('./data/chest_xray')
assert os.path.isdir(LOCAL_DATA), f'로컬 데이터 없음: {LOCAL_DATA} → download_data.ipynb 먼저'

np.random.seed(SEED); torch.manual_seed(SEED)
s3 = boto3.client('s3')

df = pd.read_csv('pneumonia_labels.csv')
test_df   = df[df['split'] == 'test'].reset_index(drop=True)
train_all = df[df['split'] == 'train'].copy()
def pkey(r):
    pid = str(r['patient_id']) if pd.notna(r['patient_id']) else ''
    return pid if pid else 'N_' + r['filename']
train_all['pkey'] = train_all.apply(pkey, axis=1)
patient_label = train_all.groupby('pkey')['label'].agg(lambda s: s.value_counts().index[0])
rng = np.random.default_rng(SEED)
val_patients = set()
for label in sorted(patient_label.unique()):
    pats = patient_label[patient_label == label].index.tolist(); rng.shuffle(pats)
    val_patients.update(pats[:max(1, int(len(pats) * VAL_RATIO))])
is_val = train_all['pkey'].isin(val_patients)
val_df   = train_all[is_val].drop(columns=['pkey']).reset_index(drop=True)
train_df = train_all[~is_val].drop(columns=['pkey']).reset_index(drop=True)
print('train', len(train_df), '/ val', len(val_df), '/ test', len(test_df))

# 클래스 가중치
import torch.nn as nn
counts = train_df['label'].value_counts().sort_index()
class_weights = torch.tensor((counts.sum()/(NUM_CLASSES*counts)).values, dtype=torch.float32).to(device)

## CELL 2 — ★★ 여기 두 줄만 바꾸세요 ★★ (실험 설정)

In [ ]:
# ===== 바꿀 곳 (이 두 줄) =====
MODEL    = 'resnet18'   # 'resnet18' 또는 'resnet34'
IMG_SIZE = 288          # 224 또는 288

# ----- 아래는 고정 (현재 최고 조합) -----
BATCH_SIZE   = 16
LR           = 1e-4
WEIGHT_DECAY = 1e-3
EPOCHS       = 13
NOTE = f'{MODEL}_{IMG_SIZE}'   # run 폴더 이름 (자동)
print('이번 실험:', NOTE, f'(batch {BATCH_SIZE}, lr {LR})')

## CELL 3 — 전처리 / 데이터로더 (IMG_SIZE 반영)

In [ ]:
import torchvision.transforms as T
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix

train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)), T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(12), T.ColorJitter(brightness=0.2, contrast=0.2),
    T.RandomResizedCrop(IMG_SIZE, scale=(0.85, 1.0)),
    T.ToTensor(), T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])

class LocalDS(Dataset):
    def __init__(self, rows, tf):
        self.rows = rows.reset_index(drop=True); self.tf = tf
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows.iloc[i]
        im = Image.open(os.path.join(LOCAL_DATA, r['relative_path'])).convert('RGB')
        return self.tf(im), int(r['label'])

train_loader = DataLoader(LocalDS(train_df, train_tf), batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader   = DataLoader(LocalDS(val_df, eval_tf),    batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
print('배치 수 — train', len(train_loader), '/ val', len(val_loader))

## CELL 4 — 모델 생성 (MODEL 반영)

In [ ]:
if MODEL == 'resnet34':
    model = models.resnet34(weights=models.ResNet34_Weights.IMAGENET1K_V1)
else:
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
print('모델:', MODEL, '| 입력:', IMG_SIZE)

## CELL 5 — 학습 루프

In [ ]:
def run_epoch(loader, train):
    model.train() if train else model.eval()
    tot, preds, labs = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if train: optimizer.zero_grad()
            out = model(imgs); loss = criterion(out, labels)
            if train: loss.backward(); optimizer.step()
            tot += loss.item()*imgs.size(0)
            preds += out.argmax(1).cpu().tolist(); labs += labels.cpu().tolist()
    return tot/len(loader.dataset), accuracy_score(labs, preds), f1_score(labs, preds, average='macro')

best_f1, history = -1, []
print(f'학습 시작 — {NOTE}, {EPOCHS} epochs\n' + '='*60)
t_all = time.time()
for epoch in range(1, EPOCHS+1):
    t0 = time.time()
    tr = run_epoch(train_loader, True)
    va = run_epoch(val_loader, False)
    scheduler.step()
    mark = ''
    if va[2] > best_f1:
        best_f1 = va[2]
        torch.save({'model_state': model.state_dict(), 'class_names': CLASS_NAMES,
                    'model_name': MODEL, 'img_size': IMG_SIZE, 'mean': MEAN, 'std': STD}, 'best_model.pth')
        mark = ' ← best 저장'
    print(f'[E{epoch:02d}/{EPOCHS}] train f1 {tr[2]:.3f} | val f1 {va[2]:.3f} ({time.time()-t0:.0f}s){mark}')
    history.append({'epoch':epoch,'train_f1':tr[2],'val_f1':va[2]})
print('='*60)
print(f'학습 완료. best val F1 = {best_f1:.4f} | 총 {(time.time()-t_all)/60:.1f}분')

## CELL 6 — test 평가

In [ ]:
ck = torch.load('best_model.pth', map_location=device); model.load_state_dict(ck['model_state']); model.eval()
test_loader = DataLoader(LocalDS(test_df, eval_tf), batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
preds, labs = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        preds += model(imgs.to(device)).argmax(1).cpu().tolist(); labs += labels.tolist()
test_acc = accuracy_score(labs, preds); test_f1 = f1_score(labs, preds, average='macro')
print(f'[{NOTE}] test 정확도 {test_acc:.4f} | test macro-F1 {test_f1:.4f}  (기준 r18_224=0.866)\n')
report = classification_report(labs, preds, target_names=CLASS_NAMES, digits=4, output_dict=True)
print(classification_report(labs, preds, target_names=CLASS_NAMES, digits=4))
cm = confusion_matrix(labs, preds)
print(pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES))

## CELL 7 — S3 저장 + 로컬 다운로드 명령

In [ ]:
run_id = time.strftime('%Y%m%d_%H%M')
run_name = f'run_{run_id}_{NOTE}'
prefix = f'{S3_DEV_ROOT}/{VERSION}/{run_name}'
config_dict = {'version':VERSION,'note':NOTE,'run_id':run_id,'split':'patientwise val resplit',
               'model':MODEL,'img_size':IMG_SIZE,'batch_size':BATCH_SIZE,'epochs':EPOCHS,
               'lr':LR,'weight_decay':WEIGHT_DECAY,'seed':SEED,'scheduler':'CosineAnnealingLR'}
metrics_dict = {'best_val_f1':round(best_f1,4),'test_acc':round(test_acc,4),
                'test_macro_f1':round(test_f1,4),'test_report':report,
                'confusion_matrix':cm.tolist(),'history':history}
with open('config.json','w') as f: json.dump(config_dict,f,ensure_ascii=False,indent=2)
with open('metrics.json','w') as f: json.dump(metrics_dict,f,ensure_ascii=False,indent=2)
for fn in ['best_model.pth','config.json','metrics.json']:
    s3.upload_file(fn, OUT_BUCKET, f'{prefix}/{fn}')
    print('업로드:', f's3://{OUT_BUCKET}/{prefix}/{fn}')

LOCAL_DEST = '/Users/skku_aws3_16/Documents/안재형/PROJECT/train/model_dev/_s3_results'
PROFILE = 'aws-say3-11'
local_run_dir = f'{LOCAL_DEST}/{VERSION}/{run_name}'
s3_run_uri = f's3://{OUT_BUCKET}/{prefix}/'
dl_cmd = f'mkdir -p "{local_run_dir}" && aws s3 cp "{s3_run_uri}" "{local_run_dir}" --recursive --profile {PROFILE} --exclude "*.pth"'
print('\n' + '='*72)
print('아래 명령을 [맥 로컬 터미널]에 붙여넣으세요 (이 run만 다운로드):')
print('='*72)
print(dl_cmd)
print('='*72)